# NB-05: adaLN-Zero Modulation and Per-Block Learned Offset

## Learning Objectives
- Understand adaLN-Zero: time conditioning produces six modulation parameters per block (shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp) via a single chunk operation
- Verify the gate=0 identity property: when gate=0, the block output equals its input — this is why adaLN-Zero enables stable training from the start
- Distinguish the DiT paper's zero-init approach from Wan2.1's actual implementation: `self.modulation = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)` uses small random initialization, not zero

## Prerequisites
- **Prior notebooks:** NB-01 (modulate function — scale/shift mechanics), NB-04 (SelfAttention — the residual branch that gates modulate)
- **Assumed concepts:** Residual connections, adaptive normalization (FiLM conditioning), why training stability matters in deep networks

## Concept Map
- adaLN-Zero six parameters → applied in DiTBlock.forward to condition self-attention and FFN branches (NB-06)
- GateModule → used in DiTBlock to gate the residual branch (NB-06)
- modulation parameter → learned per-block offset on the time embedding, unique to each of the 30 DiT blocks (NB-08)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here, _here.parent, _here.parent.parent]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
# Source: importlib.util.spec_from_file_location
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import the symbols this notebook covers
from diffsynth.models.wan_video_dit import GateModule, DiTBlock, modulate, RMSNorm

import torch
import torch.nn as nn

print("Setup complete.")

## 1. The adaLN-Zero Conditioning Mechanism

Adaptive Layer Norm (adaLN) conditions a neural network block on an external signal — in Wan2.1, that signal is the diffusion timestep embedding. The conditioning works by applying learned scale and shift to the normalized activations before the block's attention or FFN operations.

**adaLN-Zero** adds one more element: **gating**. Instead of directly adding the residual branch output, it multiplies it by a learned gate value:
```
output = x + gate * residual_branch(x)
```
When `gate = 0`, the output equals `x` exactly — the block is an identity function. This is the **zero-init property**: if gates start at zero, each block starts training as a pass-through and gradually learns to incorporate its attention/FFN branches.

**The original DiT paper** achieves gate=0 initialization by zero-initializing the final linear layer that produces the conditioning signal.

**Wan2.1** takes a different approach: a learned per-block `modulation` parameter. We will see both approaches below.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 190 and 212

## 2. GateModule — The gate=0 Identity Property

`GateModule` is the simplest module in the codebase:
```python
def forward(self, x, gate, residual):
    return x + gate * residual
```
- `gate = 0`: output = `x + 0 * residual = x` — identity (block has no effect)
- `gate = 1`: output = `x + residual` — standard residual connection (full branch incorporated)
- `gate in (0, 1)`: output interpolates between identity and full residual

**Source:** `diffsynth/models/wan_video_dit.py`, line 190

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 190 (GateModule)
B, S, dim = 1, 10, 1536
x = torch.randn(B, S, dim)             # [B, S, dim] — residual stream
residual = torch.randn(B, S, dim)       # [B, S, dim] — output of self-attention branch

# gate=0: block output == input (identity)
gate_zero = torch.zeros(B, 1, dim)      # [B, 1, dim] — broadcasts over S
out_zero = x + gate_zero * residual     # = x + 0 = x
assert torch.allclose(out_zero, x), "gate=0 must produce identity"
print(f"gate=0: block output == input (identity — training starts stable)")
print(f"Max |output - input|: {(out_zero - x).abs().max().item()}")

# gate=1: residual branch fully incorporated
gate_one = torch.ones(B, 1, dim)
out_one = x + gate_one * residual       # = x + residual
assert not torch.allclose(out_one, x), "gate=1 must differ from identity"
expected = x + residual
assert torch.allclose(out_one, expected)
print(f"gate=1: block output = input + residual (full residual connection)")
print(f"Max |output - (x+residual)|: {(out_one - expected).abs().max().item()}")

## 3. Six Modulation Parameters from DiTBlock

Each `DiTBlock` produces six modulation parameters by chunking a combined signal along the 6-parameter dimension:

```python
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
    self.modulation + t_mod
).chunk(6, dim=1)
```

- **msa parameters** (multi-head self-attention branch): `shift_msa`, `scale_msa`, `gate_msa`
  - `shift_msa`, `scale_msa` → applied to `norm1(x)` via the `modulate` function (from NB-01)
  - `gate_msa` → multiplies the self-attention branch residual
- **mlp parameters** (feed-forward network branch): `shift_mlp`, `scale_mlp`, `gate_mlp`
  - `shift_mlp`, `scale_mlp` → applied to `norm2(x)` via `modulate`
  - `gate_mlp` → multiplies the FFN branch residual

**Source:** `diffsynth/models/wan_video_dit.py`, line 212 (modulation parameter), line 219 (six-chunk operation)

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 212
dim = 1536

# Per-block learned offset — small random init (NOT zero-init like the original DiT paper)
modulation = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)  # [1, 6, dim]
print(f"modulation shape: {modulation.shape}")
print(f"modulation std: {modulation.std():.6f}")
print(f"Expected std ~ 1/sqrt(dim) = 1/sqrt({dim}) = {1/dim**0.5:.6f}")

# Zero time signal — isolates the modulation parameter's contribution
t_mod = torch.zeros(1, 6, dim)          # [1, 6, dim]
combined = modulation + t_mod           # [1, 6, dim] — just modulation itself

# Source: diffsynth/models/wan_video_dit.py, line 219
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = combined.chunk(6, dim=1)
# Each chunk: [1, 1, dim]

print("\nSix modulation parameters (t_mod=0, so values reflect only modulation init):")
for name, val in zip(
    ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"],
    [shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp]
):
    print(f"  {name}: shape={val.shape}, mean={val.mean():.6f}, std={val.std():.6f}")

## 4. Wan2.1 vs DiT Paper Initialization (Pitfall 4)

This is an important distinction that is easy to get wrong:

| Aspect | Original DiT paper | Wan2.1 |
|--------|-------------------|--------|
| Gate init | **Zero** — via zero-init of final linear layer producing the conditioning signal | **Near-zero random** — `torch.randn(1, 6, dim) / dim**0.5` |
| Block behavior at init | Exact identity (`gate = 0.0000...`) | Near-identity (gate has small random values ~0.026 for dim=1536) |
| Motivation | Guarantee stable initialization of all 30 DiT blocks simultaneously | Allow each block to learn a different learned offset from the start |

**The gate=0 identity property is still the motivating design principle** for both approaches. Wan2.1 just uses a softer version: gates start small (std ~ 1/sqrt(1536) ≈ 0.026) rather than exactly zero. Training is still stable because the gates are small enough that the blocks act approximately as identity functions at initialization.

In the cell above, you can see that `gate_msa` and `gate_mlp` have mean ≈ 0 and std ≈ 0.026 — small, but not zero.

**Source:** `diffsynth/models/wan_video_dit.py`, line 212
```python
self.modulation = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)
```

## 5. Connecting modulation Back to the modulate Function (NB-01)

The `shift_msa` and `scale_msa` parameters feed directly into the `modulate` function from NB-01. Recall that `modulate(x, shift, scale) = x * (1 + scale) + shift`. When scale and shift are small (as they are at initialization), the output is close to the input — another layer of near-identity behavior at init.

In [ ]:
# Connect adaLN parameters back to the modulate function from NB-01
# Source: diffsynth/models/wan_video_dit.py, line 64 (modulate), line 226 (DiTBlock.forward)
S = 10
x_norm = torch.randn(1, S, dim)        # [1, S, dim] — normalized activations (after LayerNorm)

# Apply shift_msa and scale_msa to the normalized residual stream
# shift_msa shape: [1, 1, dim] — broadcasts over S dimension
out_modulated = modulate(x_norm, shift_msa, scale_msa)  # x * (1 + scale_msa) + shift_msa
assert out_modulated.shape == torch.Size([1, S, dim]), f"Expected [1, {S}, {dim}], got {out_modulated.shape}"
print(f"Modulated output shape: {out_modulated.shape}")
print(f"Input std:  {x_norm.std():.4f}")
print(f"Output std: {out_modulated.std():.4f}")
print(f"Difference from identity: {(out_modulated - x_norm).abs().mean():.6f}")
print("(Small difference because scale_msa and shift_msa are near-zero at init)")

## 6. Per-Block Learned Offset — DIT-11

The `modulation` parameter is **unique to each DiTBlock instance**. Wan2.1's DiT has 30 blocks, so there are 30 separate `modulation` tensors. Each has shape `[1, 6, dim]`, giving `6 * 1536 = 9,216` learnable parameters per block for conditioning alone.

The key insight: at inference time, the combined conditioning signal is:
```python
combined = self.modulation + t_mod  # per-block offset + global time projection
```
This allows each block to **specialize its response** to the timestep embedding. Early blocks might need different scale/shift/gate values than late blocks, and the learned offset lets them diverge naturally during training.

**Source:** `diffsynth/models/wan_video_dit.py`, line 212

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 219
# Simulate: time_projection produces a signal, modulation adds per-block offset
t_mod_signal = torch.randn(1, 6, dim) * 0.1  # [1, 6, dim] — simulated time embedding projection
combined = modulation + t_mod_signal           # [1, 6, dim] — per-block offset + time signal

shift_msa2, scale_msa2, gate_msa2, shift_mlp2, scale_mlp2, gate_mlp2 = combined.chunk(6, dim=1)
print("With time conditioning signal (t_mod_signal ~ N(0, 0.01)):")
print(f"  gate_msa  mean: {gate_msa2.mean():.6f} (was {gate_msa.mean():.6f} without t_mod)")
print(f"  scale_msa mean: {scale_msa2.mean():.6f} (was {scale_msa.mean():.6f} without t_mod)")
print(f"  shift_msa mean: {shift_msa2.mean():.6f} (was {shift_msa.mean():.6f} without t_mod)")
print("\nThe time signal shifts ALL six parameters uniformly.")
print("The modulation offsets let each block respond differently to the same global t_mod.")

# Each of the 30 blocks has its own modulation — demonstrate with two different blocks
modulation_block_a = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)  # Block A
modulation_block_b = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)  # Block B
t_same = torch.randn(1, 6, dim) * 0.1  # same time signal for both blocks

gate_a = (modulation_block_a + t_same).chunk(6, dim=1)[2]  # gate_msa of block A
gate_b = (modulation_block_b + t_same).chunk(6, dim=1)[2]  # gate_msa of block B
print(f"\nSame t_mod signal, different modulation offsets:")
print(f"  Block A gate_msa mean: {gate_a.mean():.6f}")
print(f"  Block B gate_msa mean: {gate_b.mean():.6f}")
print("Blocks can specialize their conditioning via their unique modulation parameter.")

## Summary

### Key Takeaways
- **adaLN-Zero** produces six modulation parameters per block: three for self-attention (shift_msa, scale_msa, gate_msa) and three for FFN (shift_mlp, scale_mlp, gate_mlp). The gate=0 identity property means blocks can start as near-identity functions, enabling stable training of deep networks.
- **GateModule** implements `x + gate * residual`: gate=0 is identity (output == input), gate=1 is standard residual (output == input + branch_output). Verified numerically with `assert torch.allclose(out_zero, x)`.
- **Wan2.1's modulation parameter** uses small random initialization (`torch.randn(1,6,dim) / dim**0.5`), NOT zero initialization. This is a softer version of the DiT paper's exact-zero approach — blocks start near-identity rather than at exact identity. The std of gate values at init is ~1/sqrt(1536) ≈ 0.026.
- **Per-block learned offset**: Each of the 30 DiT blocks has its own `modulation` parameter (shape `[1, 6, dim]`) that adds a learned offset to the shared time conditioning signal, allowing blocks to specialize their conditioning behavior.

### Source References
| Symbol | Location |
|--------|-----------|
| GateModule | `diffsynth/models/wan_video_dit.py`, line 190 |
| DiTBlock.modulation | `diffsynth/models/wan_video_dit.py`, line 212 |
| DiTBlock.forward (six-chunk) | `diffsynth/models/wan_video_dit.py`, line 219 |

## Exercises

### Exercise 1 — Time signal effect
Change `t_mod = torch.zeros(1, 6, dim)` to `t_mod = torch.ones(1, 6, dim)` in the six-parameter extraction cell. Observe how the six parameter means shift away from the modulation-only baseline. What happens to gate_msa — is it still near zero?

### Exercise 2 — Identity block behavior
Create a GateModule instance. Pass `x=torch.randn(1,10,1536)`, `gate=torch.zeros(1,1,1536)`, `residual=torch.randn(1,10,1536)`. Verify the output equals x exactly. Then set gate to `torch.ones(1,1,1536)` and verify output equals `x + residual`.

### Exercise 3 — Modulation scale
Change the modulation initialization from `torch.randn(1, 6, dim) / dim**0.5` to `torch.randn(1, 6, dim) / dim`. Compare the std of the six parameters — how much smaller are they? What would this mean for training stability at initialization?